# Scikit-Learn Introduction for Machine Learning

## Learning Objectives
By the end of this notebook, you will be able to:
- Understand the scikit-learn API and design philosophy
- Load and explore built-in datasets
- Split data into training and testing sets
- Fit and evaluate basic ML models
- Use preprocessing and pipelines
- Perform cross-validation and hyperparameter tuning

---
## Part 1: Introduction to Scikit-Learn

**Scikit-learn** is Python's premier machine learning library:
- Simple and efficient tools for data analysis
- Consistent API across all algorithms
- Built on NumPy, SciPy, and Matplotlib

### The Estimator API
1. **Import** the estimator class
2. **Instantiate** with hyperparameters
3. **Fit** to training data
4. **Predict** or **Transform** on new data

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
%matplotlib inline

In [ ]:
# The Estimator Pattern
from sklearn.linear_model import LinearRegression

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([2, 4, 5, 4, 5])

model = LinearRegression()
model.fit(X, y)
predictions = model.predict([[6], [7]])

print(f"Coefficient: {model.coef_[0]:.3f}")
print(f"Intercept: {model.intercept_:.3f}")
print(f"Predictions: {predictions}")

---
## Part 2: Loading and Exploring Datasets

In [ ]:
from sklearn.datasets import load_iris, make_classification, make_regression

# Load Iris dataset
iris = load_iris()
print(f"Feature names: {iris.feature_names}")
print(f"Target names: {iris.target_names}")
print(f"Data shape: {iris.data.shape}")

In [ ]:
# Convert to DataFrame
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df_iris.head()

In [ ]:
# Generate synthetic data
X_clf, y_clf = make_classification(n_samples=500, n_features=2, n_redundant=0, random_state=42)
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_clf[:, 0], X_clf[:, 1], c=y_clf, cmap='RdYlBu', alpha=0.7)
axes[0].set_title('Classification Data')
axes[1].scatter(X_reg, y_reg, alpha=0.7)
axes[1].set_title('Regression Data')
plt.tight_layout()
plt.show()

---
## Part 3: Train/Test Split

**Critical**: Never evaluate on training data!

In [ ]:
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train)} samples")
print(f"Test: {len(X_test)} samples")
print(f"Class distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")

---
## Part 4: First Classification Model

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=iris.target_names, cmap='Blues', ax=ax)
ax.set_title('KNN Confusion Matrix')
plt.show()

In [ ]:
# Compare classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

classifiers = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42)
}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    print(f"{name}: {clf.score(X_test, y_test):.2%}")

---
## Part 5: Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Sample data with different scales
data = pd.DataFrame({
    'age': np.random.randint(18, 80, 100),
    'income': np.random.randint(20000, 200000, 100)
})

print("Original stats:")
print(data.describe().round(2))

scaler = StandardScaler()
scaled = scaler.fit_transform(data)
print("\nAfter StandardScaler:")
print(pd.DataFrame(scaled, columns=data.columns).describe().round(2))

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

categories = ['red', 'green', 'blue', 'red', 'blue']

le = LabelEncoder()
print(f"Label Encoding: {le.fit_transform(categories)}")

ohe = OneHotEncoder(sparse_output=False)
print(f"\nOne-Hot Encoding shape: {ohe.fit_transform(np.array(categories).reshape(-1, 1)).shape}")

In [ ]:
from sklearn.impute import SimpleImputer

data_missing = np.array([[1, 2, np.nan], [3, np.nan, 6], [7, 8, 9]])
print("With missing values:")
print(data_missing)

imputer = SimpleImputer(strategy='mean')
print("\nAfter imputation:")
print(imputer.fit_transform(data_missing))

---
## Part 6: Pipelines

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

pipe.fit(X_train, y_train)
print(f"Pipeline accuracy: {pipe.score(X_test, y_test):.2%}")

In [ ]:
from sklearn.compose import ColumnTransformer

# Mixed data example
df_mixed = pd.DataFrame({
    'age': [25, 35, 45, 55],
    'income': [50000, 75000, 100000, 60000],
    'education': ['Bachelor', 'Master', 'PhD', 'Bachelor']
})

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['age', 'income']),
    ('cat', OneHotEncoder(drop='first'), ['education'])
])

print(f"Transformed shape: {preprocessor.fit_transform(df_mixed).shape}")

---
## Part 7: Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

X, y = load_iris(return_X_y=True)
knn = KNeighborsClassifier(n_neighbors=5)

scores = cross_val_score(knn, X, y, cv=5)
print(f"5-Fold CV Scores: {scores}")
print(f"Mean: {scores.mean():.4f} (+/- {scores.std()*2:.4f})

In [ ]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    knn, X, y, cv=5,
    scoring=['accuracy', 'precision_macro', 'recall_macro'],
    return_train_score=True
)

print("Multi-metric CV Results:")
for metric in ['accuracy', 'precision_macro', 'recall_macro']:
    print(f"  {metric}: {cv_results[f'test_{metric}'].mean():.4f}")

---
## Part 8: Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance']
}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X, y)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_neighbors': randint(1, 20),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

random_search = RandomizedSearchCV(
    KNeighborsClassifier(), param_dist, n_iter=20, cv=5, random_state=42
)
random_search.fit(X, y)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")

---
## Part 9: Practice Exercises

### Exercise 1: Wine Classification
Build a complete pipeline for the Wine dataset with GridSearchCV.

In [ ]:
# Exercise 1 Solution
from sklearn.datasets import load_wine

wine = load_wine()
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    wine.data, wine.target, test_size=0.2, stratify=wine.target, random_state=42
)

pipe_wine = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])

param_grid_wine = {'svc__C': [0.1, 1, 10], 'svc__gamma': ['scale', 'auto']}
grid_wine = GridSearchCV(pipe_wine, param_grid_wine, cv=5)
grid_wine.fit(X_train_w, y_train_w)

print(f"Best params: {grid_wine.best_params_}")
print(f"Test accuracy: {grid_wine.score(X_test_w, y_test_w):.2%}")

---
## Summary

| Topic | Key Points |
|-------|------------|
| **Estimator API** | fit(), predict(), transform() |
| **Datasets** | Built-in toy datasets, synthetic generators |
| **Train/Test Split** | Always stratify for classification |
| **Preprocessing** | Scaling, encoding, imputation |
| **Pipelines** | Chain steps, prevent data leakage |
| **Cross-Validation** | Robust performance estimation |
| **Hyperparameter Tuning** | GridSearchCV, RandomizedSearchCV |

In [ ]:
# Environment verification
print("=" * 50)
print("Scikit-Learn Introduction Complete!")
print("=" * 50)
for lib in ['numpy', 'pandas', 'sklearn', 'matplotlib', 'seaborn', 'scipy']:
    try:
        mod = __import__(lib)
        print(f"✅ {lib}: {mod.__version__}")
    except:
        print(f"❌ {lib}: not available")